# Preprocessing Demonstration

This notebook demonstrates the **preprocessing utilities** shipped in `ThreeWToolkit.preprocessing`. 

Functions demonstrated:
- ImputeMissing
- Normalize
- RenameColumns
- CleanSignals
- FillLabels
- RemapClass
- SequentialPreprocessingAdapter

Importing methods:

In [ ]:
from ThreeWToolkit.preprocessing import (
    ImputeMissingConfig,
    NormalizeConfig,
    RenameColumnsConfig,
    CleanSignalsConfig,
    FillLabelsConfig,
    RemapClassConfig,
    SequentialPreprocessingAdapterConfig,
)
from ThreeWToolkit.dataset import ParquetDatasetConfig, TransformConfig

Preparing dataset for demonstration:

In [2]:
ds = ParquetDatasetConfig(path="../../dataset/", event_type=["real"]).build()

2026-04-06 16:40:20,257 | INFO | ThreeWToolkit.dataset.parquet_dataset | Dataset found at ../../dataset
2026-04-06 16:40:20,259 | INFO | ThreeWToolkit.dataset.parquet_dataset | Validating dataset integrity...
2026-04-06 16:40:20,260 | INFO | ThreeWToolkit.dataset.parquet_dataset | Dataset integrity check passed!


## Clean Signals

Here we will also introduce the **SequentialPreprocessingAdapterConfig**, which allow us to create a custom pipeline of pre-processing steps. 

- This method includes thresholds for identifying frozen / out-of-range signals based on the interquartile
range (IQR) of the average and standard deviation of the signals.

- Events whose statistics (standard deviation or average) fall outside the specified IQR thresholds may be considered faulty, and will be replaced with NaN values.

- Larger IQR thresholds will be more lenient, while smaller IQR thresholds will be more strict in identifying frozen / out-of-range signals.

- Categorical signals should not be processed by this feature extractor, as the IQR-based thresholds are designed for continuous signals.

- This method also has a **missing_columns_threshold** that will be used to drop columns that are mostly NaN values. 

In [ ]:
print(f"Columns before cleaning: {ds[0].signal.columns.tolist()}")
print(f"Original event {ds[0].signal}")

Columns before cleaning: ['ABER-CKGL', 'ABER-CKP', 'ESTADO-DHSV', 'ESTADO-M1', 'ESTADO-M2', 'ESTADO-PXO', 'ESTADO-SDV-GL', 'ESTADO-SDV-P', 'ESTADO-W1', 'ESTADO-W2', 'ESTADO-XO', 'P-ANULAR', 'P-JUS-BS', 'P-JUS-CKGL', 'P-JUS-CKP', 'P-MON-CKGL', 'P-MON-CKP', 'P-MON-SDV-P', 'P-PDG', 'PT-P', 'P-TPT', 'QBS', 'QGL', 'T-JUS-CKP', 'T-MON-CKP', 'T-PDG', 'T-TPT', 'state']
Original event                      ABER-CKGL  ABER-CKP  ESTADO-DHSV  ESTADO-M1  ESTADO-M2  \
timestamp                                                                     
2017-02-01 01:02:07        NaN       NaN          1.0        1.0        0.0   
2017-02-01 01:02:08        NaN       NaN          1.0        1.0        0.0   
2017-02-01 01:02:09        NaN       NaN          1.0        1.0        0.0   
2017-02-01 01:02:10        NaN       NaN          1.0        1.0        0.0   
2017-02-01 01:02:11        NaN       NaN          1.0        1.0        0.0   
...                        ...       ...          ...        ...    

In [42]:
clean_signal = CleanSignalsConfig()

transformer = TransformConfig(pre_processing=clean_signal).build()
transformer.fit(ds)
transformed_ds = transformer.transform(ds)

In [44]:
print(
    f"Number of columns before cleaning: {ds[0].signal.shape[1]}, after cleaning: {transformed_ds[0].signal.shape[1]}"
)

Number of columns before cleaning: 28, after cleaning: 16


In [45]:
print(f"Columns before cleaning: {transformed_ds[0].signal.columns.tolist()}")
print(f"Event after cleaning: {transformed_ds[0].signal}")

Columns before cleaning: ['ESTADO-DHSV', 'ESTADO-M1', 'ESTADO-M2', 'ESTADO-PXO', 'ESTADO-SDV-GL', 'ESTADO-SDV-P', 'ESTADO-W1', 'ESTADO-W2', 'ESTADO-XO', 'P-ANULAR', 'P-JUS-CKGL', 'P-MON-CKP', 'P-TPT', 'T-JUS-CKP', 'T-TPT', 'state']
Event after cleaning:                      ESTADO-DHSV  ESTADO-M1  ESTADO-M2  ESTADO-PXO  \
timestamp                                                            
2017-02-01 01:02:07          1.0        1.0        0.0         0.0   
2017-02-01 01:02:08          1.0        1.0        0.0         0.0   
2017-02-01 01:02:09          1.0        1.0        0.0         0.0   
2017-02-01 01:02:10          1.0        1.0        0.0         0.0   
2017-02-01 01:02:11          1.0        1.0        0.0         0.0   
...                          ...        ...        ...         ...   
2017-02-01 06:59:56          1.0        1.0        0.0         0.0   
2017-02-01 06:59:57          1.0        1.0        0.0         0.0   
2017-02-01 06:59:58          1.0        1.0   

_________

## Impute Missing Data

Function to fill missing values in a **BaseDataset** using different imputation strategies

The function accepts:  
- An imputation strategy (`strategy`) which can be:  
  - `"mean"` → fill value with the column mean  
  - `"median"` → fill value with the column median  
  - `"constant"` → fill values with a constant value
  - `"ffill"` → fill values with forward fill method
  - `"bfill"` → fill values with backward fill method 
  - `"interpolate"` → fill values with interpolate fill method 
- A constant value (`fill_value`, optional) required if the strategy is `"constant"`  

The **CleanSignals** method will be used for convenience in the following demonstrations. 

Now we will select a single event, right on we can spot a lot of *NaN* values: 

In [3]:
print(f"Original event with missing values:\n{ds[0].signal}")

Original event with missing values:
                     ABER-CKGL  ABER-CKP  ESTADO-DHSV  ESTADO-M1  ESTADO-M2  \
timestamp                                                                     
2017-02-01 01:02:07        NaN       NaN          1.0        1.0        0.0   
2017-02-01 01:02:08        NaN       NaN          1.0        1.0        0.0   
2017-02-01 01:02:09        NaN       NaN          1.0        1.0        0.0   
2017-02-01 01:02:10        NaN       NaN          1.0        1.0        0.0   
2017-02-01 01:02:11        NaN       NaN          1.0        1.0        0.0   
...                        ...       ...          ...        ...        ...   
2017-02-01 06:59:56        NaN       NaN          1.0        1.0        0.0   
2017-02-01 06:59:57        NaN       NaN          1.0        1.0        0.0   
2017-02-01 06:59:58        NaN       NaN          1.0        1.0        0.0   
2017-02-01 06:59:59        NaN       NaN          1.0        1.0        0.0   
2017-02-01 07:00

Applying `"mean"` imputation:

In [4]:
clean_signal = CleanSignalsConfig()
impute_missing = ImputeMissingConfig(strategy="mean")

transformer = TransformConfig(
    pre_processing=SequentialPreprocessingAdapterConfig(
        steps=[clean_signal, impute_missing]
    )
).build()
transformer.fit(ds)
transformed_ds = transformer.transform(ds)

In [5]:
print(f"\nValues after mean imputation: {transformed_ds[0].signal}")

ESTADO-DHSV            0.570357
ESTADO-M1              0.853289
ESTADO-M2              0.317238
ESTADO-PXO             0.008823
ESTADO-SDV-GL          0.544804
ESTADO-SDV-P           0.911002
ESTADO-W1               0.70265
ESTADO-W2              0.240788
ESTADO-XO              0.003096
P-ANULAR         14975560.96331
P-JUS-CKGL       9753444.073352
P-MON-CKP        4759819.540688
P-TPT            14256373.20793
T-JUS-CKP             62.045045
T-TPT                 111.34783
state                  0.270647
dtype: Float64

Values after mean imputation:                      ESTADO-DHSV  ESTADO-M1  ESTADO-M2  ESTADO-PXO  \
timestamp                                                            
2017-02-01 01:02:07          1.0        1.0        0.0         0.0   
2017-02-01 01:02:08          1.0        1.0        0.0         0.0   
2017-02-01 01:02:09          1.0        1.0        0.0         0.0   
2017-02-01 01:02:10          1.0        1.0        0.0         0.0   
2017-02-01 01:02:11   

________

## Normalize

Function to normalize input data using **L1**, **L2**, or **max** norm

The function accepts:  
- A norm type (`norm`) which can be:  
  - `"l1"` → normalization using the L1 norm (sum of absolute values = 1)  
  - `"l2"` → normalization using the L2 norm (Euclidean length = 1)  
  - `"max"` → normalization by dividing by the maximum absolute value  

Using column "T-TPT" to demonstrate this functionality.

Checking column before normalization:

In [6]:
ds[0].signal["T-TPT"].head(10)

timestamp
2017-02-01 01:02:07    119.0781
2017-02-01 01:02:08    119.0781
2017-02-01 01:02:09    119.0781
2017-02-01 01:02:10    119.0781
2017-02-01 01:02:11    119.0781
2017-02-01 01:02:12    119.0781
2017-02-01 01:02:13    119.0781
2017-02-01 01:02:14    119.0781
2017-02-01 01:02:15    119.0781
2017-02-01 01:02:16    119.0781
Name: T-TPT, dtype: float64

Applying L2 normalization and checking results:

In [8]:
normalize_step = NormalizeConfig(norm="l2")

transformer = TransformConfig(
    pre_processing=SequentialPreprocessingAdapterConfig(
        steps=[clean_signal, normalize_step]
    )
).build()
transformer.fit(ds)
transformed_ds = transformer.transform(ds)

In [ ]:
print(f"\nValues after normalization: {transformed_ds[0].signal['T-TPT'].head(10)}")


Values after mean imputation: timestamp
2017-02-01 01:02:07    0.736724
2017-02-01 01:02:08    0.736724
2017-02-01 01:02:09    0.736724
2017-02-01 01:02:10    0.736724
2017-02-01 01:02:11    0.736724
2017-02-01 01:02:12    0.736724
2017-02-01 01:02:13    0.736724
2017-02-01 01:02:14    0.736724
2017-02-01 01:02:15    0.736724
2017-02-01 01:02:16    0.736724
Name: T-TPT, dtype: float64


________

## Rename Columns

Function to rename columns of a DataFrame using a mapping dictionary  

The function accepts:  
- A dictionary (`columns_map`) where:  
  - Keys are the current column names  
  - Values are the new column names to assign  

### Using 3W data as example

Using the columns "ABER-CKGL" and "ABER-CKP" to demonstrate this functionality.

In [ ]:
print(f"\nOriginal columns: {ds[0].signal.columns.tolist()}")


Original columns: ['ABER-CKGL', 'ABER-CKP', 'ESTADO-DHSV', 'ESTADO-M1', 'ESTADO-M2', 'ESTADO-PXO', 'ESTADO-SDV-GL', 'ESTADO-SDV-P', 'ESTADO-W1', 'ESTADO-W2', 'ESTADO-XO', 'P-ANULAR', 'P-JUS-BS', 'P-JUS-CKGL', 'P-JUS-CKP', 'P-MON-CKGL', 'P-MON-CKP', 'P-MON-SDV-P', 'P-PDG', 'PT-P', 'P-TPT', 'QBS', 'QGL', 'T-JUS-CKP', 'T-MON-CKP', 'T-PDG', 'T-TPT', 'state']


After renaming:

In [12]:
columns_map = {"ABER-CKGL": "sensor_A", "ABER-CKP": "sensor_B"}
rename_step = RenameColumnsConfig(columns_map=columns_map)

transformer = TransformConfig(pre_processing=rename_step).build()
transformer.fit(ds)
transformed_ds = transformer.transform(ds)
print(f"\nColumns after renaming: {transformed_ds[0].signal.columns.tolist()}")


Columns after renaming: ['sensor_A', 'sensor_B', 'ESTADO-DHSV', 'ESTADO-M1', 'ESTADO-M2', 'ESTADO-PXO', 'ESTADO-SDV-GL', 'ESTADO-SDV-P', 'ESTADO-W1', 'ESTADO-W2', 'ESTADO-XO', 'P-ANULAR', 'P-JUS-BS', 'P-JUS-CKGL', 'P-JUS-CKP', 'P-MON-CKGL', 'P-MON-CKP', 'P-MON-SDV-P', 'P-PDG', 'PT-P', 'P-TPT', 'QBS', 'QGL', 'T-JUS-CKP', 'T-MON-CKP', 'T-PDG', 'T-TPT', 'state']


________

### FillLabels

In [15]:
print(f"Original signal with missing labels:\n{ds[0].label}")

Original signal with missing labels:
timestamp
2017-02-01 01:02:07    <NA>
2017-02-01 01:02:08    <NA>
2017-02-01 01:02:09    <NA>
2017-02-01 01:02:10    <NA>
2017-02-01 01:02:11    <NA>
                       ... 
2017-02-01 06:59:56       0
2017-02-01 06:59:57       0
2017-02-01 06:59:58       0
2017-02-01 06:59:59       0
2017-02-01 07:00:00       0
Name: class, Length: 21474, dtype: Int16


In [17]:
fill_labels_step = FillLabelsConfig()

transformer = TransformConfig(pre_processing=fill_labels_step).build()
transformer.fit(ds)
transformed_ds = transformer.transform(ds)

In [18]:
print(f"Event with filled labels:\n{transformed_ds[0].label}")

Event with filled labels:
timestamp
2017-02-01 01:02:07    0.0
2017-02-01 01:02:08    0.0
2017-02-01 01:02:09    0.0
2017-02-01 01:02:10    0.0
2017-02-01 01:02:11    0.0
                      ... 
2017-02-01 06:59:56    0.0
2017-02-01 06:59:57    0.0
2017-02-01 06:59:58    0.0
2017-02-01 06:59:59    0.0
2017-02-01 07:00:00    0.0
Name: class, Length: 21474, dtype: Float64


________

### RemapClass

In [33]:
print(f"Classes before remapping: {transformed_ds[594].label.unique()}")

Classes before remapping: <FloatingArray>
[0.0, 101.0, 1.0]
Length: 3, dtype: Float64


In [36]:
remap_class_step = RemapClassConfig()

transformer = TransformConfig(
    pre_processing=SequentialPreprocessingAdapterConfig(
        steps=[fill_labels_step, remap_class_step]
    )
).build()
transformer.fit(ds)
transformed_ds = transformer.transform(ds)

In [37]:
print(f"Classes after remapping: {transformed_ds[594].label.unique()}")

Classes after remapping: [ 0 10  1]
